## CNNs in Keras

### Install Keras & Tensorflow

In [0]:
!pip install --upgrade tensorflow
!pip install Keras

     |████████████████████████████████| 421.8MB 42kB/s 
     |████████████████████████████████| 3.8MB 11.6MB/s 
     |████████████████████████████████| 450kB 43.0MB/s 
  Found existing installation: tensorboard 1.15.0
    Uninstalling tensorboard-1.15.0:
      Successfully uninstalled tensorboard-1.15.0
  Found existing installation: tensorflow-estimator 1.15.1
    Uninstalling tensorflow-estimator-1.15.1:
      Successfully uninstalled tensorflow-estimator-1.15.1
  Found existing installation: tensorflow 1.15.0
    Uninstalling tensorflow-1.15.0:
      Successfully uninstalled tensorflow-1.15.0


### An example of a CNN Filter and Pooling Architecture for Natural Language Processing
<img src="cnn.png">
<div style="text-align: right"> source: “Convolutional Neural Networks for Sentence Classification”, 2014. </div>

## Load Reuter's dataset and transform labels to binary vectors

In [0]:
# Load reuter's documents and labels
import nltk
nltk.download('reuters')

from nltk.corpus import reuters
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
documents = reuters.fileids()

train = [d for d in documents if d.startswith('training/')]
test = [d for d in documents if d.startswith('test/')]

X_train = [reuters.raw(doc_id) for doc_id in train]
X_dev = [reuters.raw(doc_id) for doc_id in test]
y_train = mlb.fit_transform([reuters.categories(doc_id) for doc_id in train])
y_dev = mlb.transform([reuters.categories(doc_id) for doc_id in test])

[nltk_data] Downloading package reuters to /root/nltk_data...


## Tokenize, convert text (sequence of words) to sequence of indices and PAD the sequences

In [0]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

vocab = 20000 + 1
max_length = 1000

tokenizer = Tokenizer(num_words=vocab,oov_token='__UNK__')
tokenizer.fit_on_texts(X_train)
train_seqs = tokenizer.texts_to_sequences(X_train)
dev_seqs = tokenizer.texts_to_sequences(X_dev)
train_data = pad_sequences(train_seqs, maxlen=max_length,padding='post')
dev_data = pad_sequences(dev_seqs, maxlen=max_length,padding='post')

Using TensorFlow backend.


In [0]:
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

Found 27661 unique tokens.


In [0]:
print(train_data[0])

[ 6033   519   898  6034   730  2096     2   110     5     2  6033   519
  2028 17730     2  1477   244   361    90     6  2355  1040    10     2
  1389  3864   634  1086  8409   425    62    45    93  4342  9191  1759
     7     5    23  1225   898     2  1777   267  1091     2  3864    31
    27   529    52    21  5678    10     2   110   277    88   292    64
  2356  4491  1136     4   386  2433   520     8  1526   146    10     2
   778     4    42  1004     9   115    42  1033    25     2   323  1778
    54    21   881    13  2397    24   519  2732   158    17 10169    34
   792     5     2  5678   293  9191  1759     7   152    22   333   134
  3045    41     3   692   419  1457   472   519    22   333   658    41
  3616    39 10170   733     3    43   155    30   146  6033   472   771
   228    63    46     9  1136     6    86  5135    25  1099    63    32
     9   152    55     8   903  5136  5137  1136   333     5     2  1969
     4   603 13841   785     6  2680   152    55  4

## Download & unzip fasttext word embeddings

In [0]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
!gzip -d cc.en.300.vec.gz

--2020-03-03 22:13:45--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.vec.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 104.20.6.166, 104.20.22.166, 2606:4700:10::6814:6a6, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|104.20.6.166|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1325960915 (1.2G) [binary/octet-stream]
Saving to: ‘cc.en.300.vec.gz’

cc.en.300.vec.gz    100%[===================>]   1.23G  36.9MB/s    in 34s     

2020-03-03 22:14:20 (36.7 MB/s) - ‘cc.en.300.vec.gz’ saved [1325960915/1325960915]



## Load embeddings 

In [0]:
import numpy as np
from tqdm import tqdm

# Get the words of our vocabulary
vocab_words = list(word_index.keys())[:20000]

embeddings_index = {}

with open("cc.en.300.vec", "r", encoding="utf-8", newline="\n",errors="ignore") as f:
  for l in tqdm(f):
    values = l.split()
    if len(values) == 2:
      embedding_dim = int(values[1])
    else:
      word = values[0]
      embeddings_index[word] = np.array(values[1:]).astype(np.float)


2000001it [05:15, 6340.63it/s]


## Initialize embedding matrix with fasttext pre-trained embeddings

In [0]:
embedding_matrix = np.zeros((vocab, embedding_dim))
for word in tqdm(vocab_words):
  try:
    index = word_index[word]
    embedding_matrix[index] = embeddings_index[word]
  except:
    pass

100%|██████████| 20000/20000 [00:00<00:00, 391604.91it/s]


In [0]:
embedding_matrix.shape

(20001, 300)

In [0]:
np.save("embedding_matrix.npy", embedding_matrix, allow_pickle=True)

### Evaluation metrics

In [0]:
def recall(y_true, y_pred):
    
    """
    Recall metric.
    Only computes a batch-wise average of recall.
    Computes the recall, a metric for multi-label classification of
    how many relevant items are selected.
    """
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
    recall = true_positives / (possible_positives + K.epsilon())
    return recall


def precision(y_true, y_pred):
    
    """
    Precision metric.
    Only computes a batch-wise average of precision.
    Computes the precision, a metric for multi-label classification of
    how many selected items are relevant.
    Source
    ------
    https://github.com/fchollet/keras/issues/5400#issuecomment-314747992
    """
    true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
    predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
    precision = true_positives / (predicted_positives + K.epsilon())
    return precision


def f1(y_true, y_pred):
    
    """Calculate the F1 score."""
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * ((p * r) / (p + r))


def accuracy(y_true, y_pred):
    return K.mean(K.equal(y_true, K.round(y_pred)), axis=1)

## CNN model

In [0]:
#Create and train a CNN model with trigram filters

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
import warnings
import sklearn.exceptions
warnings.filterwarnings("ignore", category=sklearn.exceptions.UndefinedMetricWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
from keras.callbacks import ModelCheckpoint
from keras.layers import Dense, Activation, Dropout, Flatten
from keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D 
from keras.models import Sequential
from keras.optimizers import Adam
from keras import backend as K

FILTERS = 128
KERNEL = 3
DENSE = 256
N_CLASSES = 90

# create empty sequential model
model = Sequential()
# add an embedding layer
model.add(Embedding(vocab, embedding_dim, weights=[embedding_matrix], 
                    input_length=max_length, trainable=False))
# add 0.2 dropout probabillity
model.add(Dropout(0.2))
# add a convolution layer 
model.add(Conv1D(FILTERS, KERNEL, activation='relu', padding='valid'))
# max pooling with pool size=2
model.add(MaxPooling1D(2))
# add another convolution layer 
model.add(Conv1D(FILTERS, KERNEL, activation='relu', padding='valid'))
# max pooling
model.add(GlobalMaxPooling1D())
# add 0.2 dropout probabillity
model.add(Dropout(0.2))
# add dense layer
model.add(Dense( DENSE, activation='relu' ))
# add final linear layer
model.add(Dense( N_CLASSES, activation='sigmoid' ))

print(model.summary())
model.compile(loss='binary_crossentropy',
                  optimizer=Adam(lr=0.001),
                  metrics=[precision, recall, f1, accuracy])

checkpoint = ModelCheckpoint('keras_CNN_model', monitor='val_f1', verbose=1, save_best_only=True, mode='max')

model.fit(train_data, y_train,
              batch_size=32,
              epochs=5,
              verbose = 2,
              callbacks=[checkpoint],
              validation_data=(dev_data, y_dev),
              shuffle=True)

Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (None, 1000, 300)         6000300   
_________________________________________________________________
dropout_3 (Dropout)          (None, 1000, 300)         0         
_________________________________________________________________
conv1d_3 (Conv1D)            (None, 998, 128)          115328    
_________________________________________________________________
max_pooling1d_2 (MaxPooling1 (None, 499, 128)          0         
_________________________________________________________________
conv1d_4 (Conv1D)            (None, 497, 128)          49280     
_________________________________________________________________
global_max_pooling1d_2 (Glob (None, 128)               0         
_________________________________________________________________
dropout_4 (Dropout)          (None, 128)              

In [0]:
#Create and train a CNN model with (2,3,4)-gram filters using Keras functional API

import warnings
import sklearn.exceptions
warnings.filterwarnings("ignore", category=sklearn.exceptions.UndefinedMetricWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
from keras.callbacks import ModelCheckpoint
from keras.layers import Dense, Input, Activation, Dropout, Flatten
from keras.layers import Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D, concatenate
from keras.models import Model
from keras.optimizers import Adam
from keras import backend as K 


embed_layer = Embedding(vocab, embedding_dim, weights=[embedding_matrix],
          input_length=max_length, trainable=False)

FILTERS = 128
DENSE = 256
N_CLASSES = 90

convs = []
filter_sizes = [2,3,4]

sequence_input = Input(shape=(max_length,), dtype='int32')
embedded_sequences = embed_layer(sequence_input)

for size in filter_sizes:
    embedded_sequences = Dropout(0.2)(embedded_sequences)
    l_conv = Conv1D(activation='relu',filters=FILTERS, kernel_size=size)(embedded_sequences)
    l_pool = MaxPooling1D(4)(l_conv)
    convs.append(l_pool)
    
l_merge = concatenate(convs,axis=-1)
l_cov1= Conv1D(activation='relu',filters=FILTERS, kernel_size=4)(l_merge)
l_pool1 = MaxPooling1D(4)(l_cov1)
l_cov2 = Conv1D(activation='relu',filters=FILTERS, kernel_size=4)(l_pool1)
l_pool2 = GlobalMaxPooling1D()(l_cov2)
l_pool2 = Dropout(0.2)(l_pool2)
l_dense = Dense(DENSE, activation='relu')(l_pool2)
probs = Dense(N_CLASSES, activation='sigmoid')(l_dense)

model = Model(sequence_input, probs)

model.compile(loss='binary_crossentropy',
                  optimizer=Adam(lr=0.001),
                  metrics=[precision, recall, f1, accuracy])

print(model.summary())

checkpoint = ModelCheckpoint('keras_Deep_CNN_model', monitor='val_f1', verbose=1, save_best_only=True, mode='max')

model.fit(train_data,y_train,
              batch_size=32,
              epochs=5,
              verbose = 2,
              callbacks=[checkpoint],
              validation_data=(dev_data,y_dev),
              shuffle=True)

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            (None, 1000)         0                                            
__________________________________________________________________________________________________
embedding_3 (Embedding)         (None, 1000, 300)    6000300     input_1[0][0]                    
__________________________________________________________________________________________________
dropout_5 (Dropout)             (None, 1000, 300)    0           embedding_3[0][0]                
__________________________________________________________________________________________________
dropout_6 (Dropout)             (None, 1000, 300)    0           dropout_5[0][0]                  
____________________________________________________________________________________________